READ DATABASE

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/bigmart/')

In [0]:
df = (spark.read.format('csv')
      .option('inferSchema', True)
      .option('header', True)
      .load('/Volumes/workspace/default/bigmart/BigMart Sales.csv'))

display(df)

In [0]:
df.printSchema()      # column names and data types
print(df.count())     # number of rows
df.describe().show()  # basic statistics

In [0]:
df.display()

DATA READING JSON

In [0]:
df_json = (spark.read.format('json')\
           .option('header', True)
           .option('multiLine', False)
           .load('/Volumes/workspace/default/bigmart/drivers.json'))

display(df_json)

SCHEMA

In [0]:
df.printSchema()

In [0]:
df_json.printSchema()
print(df_json.count())

DDL COMMAND

In [0]:
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df = spark.read.format('csv')\
        .schema(my_ddl_schema)\
        .option('header', True)\
        .load('/Volumes/workspace/default/bigmart/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

STRUCTTYPE() SCHEMA

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_strct_schema = StructType([
                                StructField('Item_Identifier',StringType(),True),
                                StructField('Item_Weight',StringType(),True),
                                StructField('Item_Fat_Content',StringType(),True),
                                StructField('Item_Visibility',StringType(),True),
                                StructField('Item_MRP',StringType(),True),
                                StructField('Outlet_Identifier',StringType(),True),
                                StructField('Outlet_Establishment_Year',StringType(),True),
                                StructField('Outlet_Size',StringType(),True),
                                StructField('Outlet_Location_Type',StringType(),True),
                                StructField('Outlet_Type',StringType(),True),
                                StructField('Item_Outlet_Sales',StringType(),True)

])

In [0]:
df = spark.read.format('csv')\
        .schema(my_strct_schema)\
        .option('header', True)\
        .load('/Volumes/workspace/default/bigmart/BigMart Sales.csv')

In [0]:
df.printSchema

SELECT


In [0]:
df.display()

In [0]:
df_sel = df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

### ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

In [0]:
df.display()

### FILTERS

Scenario-1

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

Scenario-2

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight').cast('double') < 10)).display()

Scenario-3

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

###WITHCOLUMN RENAME

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

###WITHCOLUMN

Scenario-1

In [0]:
df = df.withColumn('Flag',lit("new")).display()

Scenario-2

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

Scenario-3

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","LF")).display()

### TYPE CASTING

In [0]:
df = df.withColumn('Item_Weight',col('Item_Weight').cast('string'))

In [0]:
df.printSchema()

###SORTING

Scenario-1

In [0]:
df.sort(col('Item_Weight').desc()).display()

Scenario-2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

Scenario-3

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[0,0]).display()

Scenario-4

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[0,1]).display()

###LIMIT

In [0]:
df.limit(10).display()

###DROP

Scenario-1

In [0]:
df.drop('Item_Visibility').display()

Scenario-2

In [0]:
df.drop('Item_Visibility','Outlet_Type').display()

###DROP-DUPLICATES

Scenario-1

In [0]:
df.dropDuplicates().display()

Scenario-2

In [0]:
df.drop_duplicates(subset = ['Outlet_Type']).display()

In [0]:
df.distinct().display()

###UNION AND UNION BYNAME

Preparing Dataframe

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)


In [0]:
df1.display()

In [0]:
df2.display()

###UNION

In [0]:
df1.union(df2).display()

###UNION BYNAME

In [0]:
data1 = [('kad','1',),
        ('sid','2',)]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)

df1.display()

In [0]:
df1.unionByName(df2).display()

###STRING FUNCTIONS

INITCAP()

In [0]:
df.select(initcap('Item_MRP')).display()

In [0]:
df.select(lower('Item_MRP')).display()

In [0]:
df.select(upper('Item_MRP')).display()

In [0]:
df.select(upper('Item_MRP').alias('Item_type')).display().display()

###DATE FUNCTION

Current_Date

In [0]:
df = df.withColumn('curr_date',current_date())
df.display()

Date_Add


In [0]:
df = df.withColumn('week_after',date_add('curr_date',7))
df.display()

Date_sub

In [0]:
df.withColumn('week_befor',date_sub('curr_date',7)).display()

In [0]:
df = df.withColumn('week_befor',date_add('curr_date',-7))
df.display()

###DATEDIF

In [0]:
df = df.withColumn('dateif',datediff('curr_date','week_after'))
df.display()

###DATE FORMATE

In [0]:
df = df.withColumn('week_befor',date_format('week_befor','dd-MM-yyyy'))
df.display()

###HANDLING NULLS

DROPPING NULLS

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_size']).display()

FILLING NULLS

In [0]:
df.fillna(0).display()

In [0]:
df.fillna('Unknown', subset=['Outlet_Size']).display()

### SPLIT AND INDEXING

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

###INDEXING

In [0]:
df.withColumn('Item_Outlet_Sales',split('Item_Outlet_Sales',' ')[1]).display()

#EXPLODE

In [0]:
df_exp = df.withColumn('Item_Outlet_Sales',split('Item_Outlet_Sales',' '))

df_exp.display()

In [0]:
df_exp.withColumn('Item_Outlet_Sales',explode('Item_Outlet_Sales')).display()

###ARRAY-CONTAINS

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Item_Outlet_Sales','Type1')).display()

###GROUP BY

Scenario-1

In [0]:
df.groupBy('Item_MRP').agg(sum('Outlet_Identifier')).display()

In [0]:
df.groupBy('Item_MRP').agg(avg('Outlet_Identifier')).display()

Scenario-2

In [0]:
df.groupBy('Item_MRP','Outlet_Location_Type').agg(sum('Outlet_Identifier').alias('Total_MRP')).display()

Scenario-4

In [0]:
df.groupBy('Item_MRP','Outlet_Location_Type').agg(sum('Outlet_Identifier'),avg('Item_Weight')).display()

###COLLECT_LIST

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

###PIVOT

In [0]:
df.groupBy('Item_MRP').pivot("Outlet_Location_Type").agg(avg("Outlet_Identifier")).display()

###WITH-OTHERWISE

In [0]:
df.withColumn('veg_flag',when(col('Item_MRP')=='Meat','Non-veg').otherwise('veg')).display()

Scenario-1

In [0]:
df.withColumn('veg_flag', when(col('Item_MRP')=='Meat','Non-veg').otherwise('veg')).withColumn('veg_flag', when((col('veg_flag')=='veg') & (col('outlet_Identifier')<100.0), 'Veg_Inexpensive').when((col('veg_flag')=='veg') & (col('outlet_Identifier')>100.0), 'Veg_Expensive')).display() 

###JOINS

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

###INNER JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()

###LEFT JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

###RIGHT JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

###ANTI JOIN

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

###WINDOW FUNCTIONS

###ROW NUMBER

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

###RANK

In [0]:
df.withColumn('rank',rank().over(Window.orderBy('Item_Identifier'))).display()

###DENSE_RANK

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

###RANK VS DESNSE RANK

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

###CUMULATIVE SUM

In [0]:
df.withColumn('cumsum',sum('Outlet_Identifier').over(Window.orderBy('Item_MRP'))).display()

In [0]:
df.withColumn('cumsum',sum('Outlet_Identifier').over(Window.orderBy('Item_MRP').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

###USER DEFINED FUNCTION

#####STEP-1

In [0]:
def my_func(x):
    return x*x

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol',my_udf(col('Outlet_Identifier').cast('double'))).display()

###DATA WRITING

#####CSV

In [0]:
df.write.format('csv')\
    .save('/Volumes/workspace/default/bigmart/CSV/data.csv')

###APPEND

In [0]:
df.write.format('csv')\
        .mode('append')\
        .save('/Volumes/workspace/default/bigmart/CSV/data.csv')

In [0]:
df.write.format('csv')\
        .mode('append')\
        .option('path','/Volumes/workspace/default/bigmart/CSV/data.csv')\
        .save()

###OVERWITE

In [0]:
df.write.format('csv')\
.mode('overwrite')\
.option('path','/Volumes/workspace/default/bigmart/CSV/data.csv')\
.save()

###ERROR

In [0]:
df.write.format('csv')\
.mode('overwrite')\
.option('path','/Volumes/workspace/default/bigmart/CSV/data.csv')\
.save()

###IGNORE

In [0]:
df.write.format('csv')\
.mode('ignore')\
.option('path','/Volumes/workspace/default/bigmart/CSV/data.csv')\
.save()

###PARQUET FILE FORMAT

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.option('path','/Volumes/workspace/default/bigmart/parquet/data.parquet')\
.save()

###MANAGED VS EXTERNAL TABLES

In [0]:
df.write.format('delta')\
.mode('overwrite')\
.saveAsTable('my_table')

In [0]:
df.display()

###SPARK SQL

###CREATE TEMPORARY VIEW

In [0]:
df.createTempView('my_view')

In [0]:
%sql

select * from my_view

In [0]:
df_sql = spark.sql("select * from my_view")

In [0]:
df_sql.display()